In [ ]:
import nibabel as nib
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from dipy.segment.mask import median_otsu
from torch.utils.data import Dataset, DataLoader
from matplotlib import pyplot as plt
import matplotlib.pyplot as plt
import time

In [ ]:
# Check if GPU is available
if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))
elif device.type == "mps":
    print("Apple Metal GPU available")

In [ ]:
##loding data
path = "/Users/sarojrai/Desktop/LSBU/Dissertation /IVIM_Autoencoder/notebooks/BRAIN"

raw_data=nib.load(path+"/data.nii.gz")
mri_data=raw_data.get_fdata()
print(mri_data.shape)

## **Data description**
1. H: Height of one image in the sequence
2. W: Width of one image in the sequence
3. D: Depth of images or Number of images in the sequence
4. C: Number of channels (like 3 RGB or b-values)

In [ ]:
# data = torch.randn(2,3,4,4)
# data.shape
# data[0]
# plt.imshow(data[0])

In [ ]:
##Data Splitting 70% for train and 30% for test data
slice_indx=int(0.57*mri_data.shape[2])

##train and test
train = mri_data[:,:,:slice_indx,:]
test = mri_data[:,:,slice_indx:,:]

In [ ]:
slice_indx

In [ ]:
##displaying the 2 slice of mri data over 21 b-value
fig, ax = plt.subplots(3,7, figsize=(10,6))

# Flatten the 2D array of axes into a 1D array for easier iteration
flat_axes = ax.ravel()

plot_counter = 0
# Iterate over the first two slices (depth dimension) of the MRI data
for i in range(train[:,:,:2,:].shape[2]):
  # Iterate over all b-values
  for j in range(train.shape[3]): # j will be 0 to 20 b-values
    if plot_counter < len(flat_axes): # Check if there are still subplots available
      current_ax = flat_axes[plot_counter]
      current_ax.imshow(train[:,:,i,j], cmap='gray')
      current_ax.set_title(f"Slice {i}, b={j}") # Add a title for context
      current_ax.axis('off') # Turn off axis labels for cleaner display
      plot_counter += 1

plt.tight_layout() # Adjust subplot parameters for a tight layout
plt.show()

In [ ]:
img = train[:,:,20,0]
#for standardize the pixel values
# mean = np.mean(img)
# std = np.std(img)
# img = img-mean
# img = img/std
plt.hist(img.flatten(),bins=200)
plt.show()


In [ ]:
# Extract b0 directly from raw mri_data (H, W, D, b) order — no transpose needed
b0_raw = train[:,:, 0]  # shape (256, 256, 54) = (H, W, D)

_, otsu_mask = median_otsu(b0_raw, median_radius=3, numpass=2)
brain_msk = otsu_mask.astype(np.uint8)   #shape (256, 256, 54) = (H, W, D)

print("Mask shape (H, W, b-value):", brain_msk.shape)  #(256, 256, 54)

In [ ]:
d=train[:,:,2,:]
print(f"data shape {np.transpose(d, (2,0,1)).shape}")
print(f"Mask shape {np.transpose(brain_msk).shape}")

## **DATA PREPROCESSING FOR SIMILAR NEIGHBOURING PIXELS EXTRACTION**

In [ ]:
def process_chunk(padded_px, padded_b0, y, x, dy, dx,
                    center_flat_idx, target_slots, b_value):
    """
    Broadcasting Technique:
      Vectorized core, run on one chunk of pixels at a time so peak memory
    stays bounded regardless of total voxel count N.
    """
    n = len(y)

    ##Broadcasting technique to speed up the calculation
    # Build 5x5 neighborhood (25 positions per pixel) fetch using 5x5 window
    yy = y[:, None] + dy[None, :]
    xx = x[:, None] + dx[None, :]

    ##similar neighbouring pixels calculation
    b0_neighbors = padded_b0[yy, xx]              # (n, 25)
    center_b0 = padded_b0[y, x]                     # (n,)
    similarity = np.abs(center_b0[:, None] - b0_neighbors)  # (n, 25)

    ##a masking array of 25 boolean coordinate
    keep = np.ones(25, dtype=bool)

    ##removing center coordinate from the extracted 5x5 similarity matrix
    keep[center_flat_idx] = False

    ##indexing drops/remove center from the column where keep==false
    similarity = similarity[:, keep] # (n, 24)

    # Drops the center coordinates
    yy, xx = yy[:, keep], xx[:, keep]

    #Stable sort to match Python's stable list.sort()
    order = np.argsort(similarity, axis=1, kind='stable')  # (n, 24)

    ##order indices says which column to pick per row
    top8_indices = order[:, :8]

    ##selects values from an array along a specified axis using index arrays
    yy_top = np.take_along_axis(yy, top8_indices, axis=1)      # (n, 8)
    xx_top = np.take_along_axis(xx, top8_indices, axis=1)
    full_top8 = padded_px[:, yy_top, xx_top]  # (b_value, n, 8)

    ##initialize the 3x3 patch of most similar pixels
    block = np.zeros((n, 3, 3, b_value), dtype=np.float32)
    center_vals = padded_px[:, y, x]             # (b_value, n)
    block[:, 1, 1, :] = center_vals.T

    vals = np.transpose(full_top8, (1, 2, 0))           # (n, 8, b_value)
    block[:, target_slots[:, 0], target_slots[:, 1], :] = vals

    return block

In [ ]:
def similar_neighborhood_patch(data, brain_masker, chunk_size=20000):
    """
    For each depth slice, extracts every brain pixels 5x5 neighborhood, selects
    the 8 pixels most similar to the center in b0-intensity, and packs them
    (in similarity-rank order) into a 3x3 block with the center pixel at (1,1).

    Processes voxels in chunks of `chunk_size` to bound peak memory.

    Parameters
    ----------
    mri_data : ndarray, shape (H, W, D, Num_b)
        4D MRI data with spatial dimensions Height, Width, Depth and
        Num_b b-value measurements. The first measurement (index 0) is
        assumed to be the b=0 image.
    brain_mask : ndarray, shape (H, W, b-value)
        3D binary brain mask. pixels with value > 0 are considered brain.
    b0_threshold : float, default 1.0
        Minimum b0 intensity for a pixels to be included. pixels with
        b0_image < b0_threshold are excluded even if inside the mask.
    chunk_size : int, default 20000
        Number of pixel processed per chunk to limit memory usage.
    """
    ##Shape of the raw data
    H, W, D, b_value = data.shape

    ##x and y factors to normalize the coordinates
    y_factor = 2.0 / (H - 1) if H > 1 else 1.0
    x_factor = 2.0 / (W - 1) if W > 1 else 1.0

    ##initializing empty list for the final data
    similar_patch, center_coord, norm_cent_coord  = [], [], []

    ##2D slices from the 3D MRI data
    for i in range(data.shape[2]):
      slices_data = data[:,:,i,:]

      ##pytorch convention (Channel, H,W,D)
      mri_slice = np.transpose(slices_data, (2, 0, 1))  # (b, H, W)


      # brain_mask must be transposed the same way as mri_data,
      brain_mask_slice = np.transpose(brain_masker, (2,0,1)) # (b, H, W)

      ##mean of valid brain intensity
      b0_img_mean = np.mean(brain_mask_slice>0)

      ##threshold
      b0_threshold = 0.50*b0_img_mean

      ##first image at b-value=0
      b0_image = mri_slice[0]

      ##brain indices selects the brain pixels to process
      brain_indices = np.argwhere((brain_mask_slice > 0) & (b0_image >= b0_threshold))
      N = len(brain_indices)
      print(f"Processing {N} brain pixel to build 2D similarity blocks...")

      ##padding tensors
      padded_px = np.pad(mri_slice, ((0, 0), (2, 2), (2, 2)), mode='constant', constant_values=0)
      ##padded signal at b-value=0
      padded_b0 = padded_px[0]

      ##x and y coordinates of brain indices with padding
      y_all = (brain_indices[:, 0] + 2).astype(np.int32)
      x_all = (brain_indices[:, 1] + 2).astype(np.int32)

      ##array ranging from -2 to 3, for generating a 5x5 window: array([-2, -1,  0,  1,  2])
      offs = np.arange(-2, 3, dtype=np.int32)

      ##5x5 window matrix
      dy, dx = np.meshgrid(offs, offs, indexing='ij')
      ##flatten them in 1d array
      dy, dx = dy.ravel(), dx.ravel()
      ##offset coordinate index for central pixel
      center_flat_idx = 12  #(0,0) offset's position in the flatten 25-length grid

      ##initialize the empty neighbors slot to store top neighbor pixels later in calculation
      top_neighbors_slot = np.array([
          (a, b) for a in range(3) for b in range(3) if not (a == 1 and b == 1)
      ])  #(8, 2)

      ##initialize the empty similar neighbourhood block array
      similar_neig_block = np.zeros((N, 3, 3, b_value), dtype=np.float32)

      ##actual similar neighbourhood data calculated in chunks and store in pre-defined similar_neig_block
      for start in range(0, N, chunk_size):
          end = min(start + chunk_size, N)
          similar_neig_block[start:end] = process_chunk(padded_px, padded_b0, y_all[start:end], x_all[start:end],
                                                    dy, dx, center_flat_idx, top_neighbors_slot, b_value)

      center_coords = list(map(tuple, brain_indices))
      y0 = brain_indices[:, 0] * y_factor - 1.0
      x0 = brain_indices[:, 1] * x_factor - 1.0
      center_norm_coord = list(zip(y0.tolist(), x0.tolist()))

      similar_patch.append(similar_neig_block)
      center_coord.append(center_coords)
      norm_cent_coord.append(center_norm_coord)
    print("Processing completed.")
    return similar_patch, center_coord, norm_cent_coord

In [ ]:
# Load b-values from the file
with open(path+"/bvalues.bval", "r") as f:
    bvals_str = f.read().split()
bvals_arr = np.array([float(b) for b in bvals_str], dtype=np.float32)

print(f"Loaded b-values: {bvals_arr}")
print(f"Number of b-values (N_b): {len(bvals_arr)}")

In [ ]:
# Now this matches what similar_neighborhood_3d_fast expects internally:
similar_patch, coords, norm_coords = similar_neighborhood_patch(
    train, brain_msk)

In [ ]:
# Run similar_neighborhood_search with actual MRI data
print(f"Extracted {len(similar_patch)} patches from real data")
print("Example block shape:", similar_patch[1].shape)
# print("Example normalised coordinate:", norm_coords[0])

In [ ]:
##display 3x3 block
plt.imshow(similar_patch[0][0,:,:,0], cmap='gray') # Display the signal at 21 b-value of the first block's middle depth slice
plt.show()

In [ ]:
similar_patch[0] ##shape=(119110, 3, 3, 21)

In [ ]:
x=torch.from_numpy(similar_patch[0]).float()
x.shape

In [ ]:
##shape=(119110, 3, 3, 21) -----> shape=(119110, 21, 3, 3) ----> Idx shape=(0, 3, 1, 2)
x1=x.permute(2, 0, 1) 
x1.shape

In [ ]:
print(x1[:,0:1, :, : ])
x1[:, 0:1, :, : ].shape ##torch.Size([21, 3, 3])

In [ ]:
x1[:,0:1,:,:][0].shape

In [ ]:
##b0_image
plt.imshow(x1[:, 0:1, :, :][0], cmap='gray')
plt.show()

In [ ]:
# Visualise the highest b-value (e.g., index -1), with non-zero b-value
plt.imshow(x1[0, -1, :, :], cmap="gray", vmin=0, vmax=1)
plt.colorbar()
plt.show()

In [ ]:
len(norm_coords)

## **DATASET**

In [ ]:
# Test a batch from the real data dataloader
class DiffusionDataset(Dataset):
    def __init__(self, similar_patches, condition):
        self.patches = similar_patches
        self.condition = condition

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.patches[idx]).float()

        # For a single patch, shape is typically (3, 3, b-values)
        # and should become (b-values, 3, 3) for the model.
        # If a batch is passed in instead, keep the batch dimension first.
        if x.dim() == 3:
            x = x.permute(2, 0, 1)
        elif x.dim() == 4:
            x = x.permute(0, 3, 1, 2)
        else:
            raise ValueError(f"Unexpected tensor shape {tuple(x.shape)}")

        y = torch.tensor(self.condition[idx]).float()
        return x, y

In [ ]:
np.array([item for sublist in norm_coords for item in sublist])

In [ ]:
##Batch Size
BATCH_SIZE = 64

# Flatten similar_patch and norm_coords before passing to Dataset
flat_similar_patches = np.concatenate(similar_patch, axis=0)
flat_norm_coords = np.array([item for sublist in norm_coords for item in sublist])

# Initialize pipeline matching your batch configuration
dataset_real = DiffusionDataset(flat_similar_patches, flat_norm_coords)

# DataLoader will combine the patches into a batch tensor
# Expected input to model: (Batch, Channels, Height, Width)
dataloader_real = DataLoader(
    dataset=dataset_real,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
)

print(f"Dataset size: {len(dataset_real)}")
print(f"First batch shapes: {next(iter(dataloader_real))[0].shape}, {next(iter(dataloader_real))[1].shape}")

In [ ]:
# for batch_x, batch_c in dataloader_real:
#     print(f"Batch x: {batch_x.shape} and Coordinates: {batch_c.shape}")

## **PHYSIOLOGICAL BI-EXPONENTIAL SIGNAL EQUATION**

In [ ]:
##Physics IVIM signal function
def ivim_biex_signal(theta, b_vals):
    ##IVIM parameters
    D = theta[:, 0:1]
    Ds = theta[:, 1:2]
    f = theta[:, 2:3]

    #b values broadcastable if b-value is 1D array
    if b_vals.dim()<=2:
        b = b_vals.view(1,-1, 1, 1)
    else:
        b = b_vals.reshape(1, -1, 1, 1)
    
    ##perfusion signal due to the flow of blood
    s_perf = f*torch.exp(-b * Ds)
    ##True diffusion signal due to water molecules in tissues
    s_diff = (1-f)*torch.exp(-b*D)
    ##bi-exponential signal
    s_sum = s_perf + s_diff
    return s_sum, s_perf, s_diff

## **MODEL ARCHITECTURE AND LOSS**

In [ ]:
##Physics informed IVIM cVAE
class PhysicsCVAE(nn.Module):
    """
    Physics-Informed Conditional Variational Autoencoder (PIcVAE) for IVIM parameter estimation.

    Encoder: 2D Convolutional Layer for 3x3 patches
    Decoder: z+Coordinates per pixel
    """
    def __init__(self, in_channel:int, bvalues: torch.Tensor, latent_dim=64, coord_dim=2, bvals=None):
        """
        PARAMETERS:
        in_channel: Input channel based on the number of b-values.
        latent_dim: Dimensionality of z.
        """
        super().__init__()
        self.latent_dim = latent_dim
        self.coord_dim = coord_dim

        ##registering b-values as buffer parameters
        self.register_buffer("bvalues", bvalues.view(1, -1, 1, 1)) 

        ##Encoder layer
        self.encoder_net = nn.Sequential(
            ##first conv layer
            nn.Conv2d(in_channel, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Dropout2d(0.25),

            ##second conv layer
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            # nn.Dropout2d(0.10),

            ##third conv layer
            nn.Conv2d(128, 96, kernel_size=3, padding=1),
            nn.BatchNorm2d(96),
            nn.ReLU()
        )

        # Flattened size: 32 filters * depth * H=1 * W=1
        self.encoder_fc = nn.Sequential(
            nn.Linear(96*3*3, 64),
            nn.ReLU(),
        )

        ##mean and variance for latent space
        self.fc_mu = nn.Linear(64, latent_dim)
        self.fc_logvar = nn.Linear(64, latent_dim)

        ##
        self.decoder_net = nn.Sequential(
            nn.Linear(latent_dim + self.coord_dim, 128), 
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 3*3*3) ##3 parameters
        )
        
    # Physical parameter constraints
    @staticmethod
    def physics_constraint(raw_params):
        """
        Split (B, 27) → 3 param maps with physical constraints.
        Returns: theta (B, 3, 3, 3) stacked as [D, D*, f]
                Individual: D (B,1,3,3), Ds (B,1,3,3), f (B,1,3,3)
        """
        ## Physiological Constant ranges
        D_MIN,  D_MAX   = 0.0001, 0.004 #Water restriction boundaries
        Ds_MIN, Ds_MAX  = 0.005, 0.020 #High-velocity capillary perfusion micro-circulation
        f_MIN,  f_MAX   = 0.000, 0.300 #Capillary plasma fraction cap (50%)

        ##splitting the raw params into 3 groups
        d_chunk, ds_chunk, f_chunk = torch.chunk(raw_params, chunks=3, dim=1)

        #Reshape to 2D spatial patches: Each becomes (B, 1, 3, 3)
        d_raw = d_chunk.view(-1, 1, 3, 3)
        ds_raw = ds_chunk.view(-1, 1, 3, 3)
        f_raw = f_chunk.view(-1, 1, 3, 3)

        ##IVIM THREE PARAMETERS
        ##True diffusion
        ##True and Pseudo - diffusion value is capped at maximum of 1.0
        t_D = torch.clamp(F.softplus(d_raw), max=1.0)

        ##Perfusion - pseudo-diffusion
        p_D = torch.clamp(F.softplus(ds_raw), max=1.0)

        ##perfusion fraction: Perfusion fraction remains a standard native sigmoid percentage 
        p_f = torch.sigmoid(f_raw)

        ##Scaling the three parameters using biophysiological constants 
        D  = D_MIN + t_D*(D_MAX - D_MIN)
        Ds = Ds_MIN + p_D*(Ds_MAX - Ds_MIN)
        f  = f_MIN + p_f*(f_MAX - f_MIN)

        ##Stacking parameters safely back into the (B, 3, 3, 3) shape
        theta_params = torch.cat([D, Ds, f], dim=1)

        return D, Ds, f, theta_params

    ## CVAE methods
    ##ENCODER
    def Encode(self, x):
        "The x has (batch, b-value, H, W) that generates the mu, logvar"
        h1 = self.encoder_net(x)
        ##keep the batch dimension as it is, and flatten everything else
        h2 = h1.view(h1.size(0), -1) 
        h3 = self.encoder_fc(h2)
        mu = self.fc_mu(h3)
        logvar = self.fc_logvar(h3)
        return mu, logvar

    ##LATENT SPACE
    ##Appying reparameterization trick: way to sample the latent variable(z) from the distribution of N(mu, variance)
    def reparameterize(self, mu, logvar):
        ##standard deviation from log-variance
        std = torch.exp(0.5 * logvar) 
        ##epsilon is a noise variable with randomness on it
        eps = torch.randn_like(std) 
        ##reparameterized sample
        z = mu + eps*std
        return z
    
    ##DECODER
    def Decode(self, z_sample, coord):
        """
        z_sample: (B, latent_dim)
        coord: (B,2)
        Returns: signal(B, b-values, 3,3)
        """
        ##decoder input
        de_input = torch.cat([z_sample, coord], dim=1) ##(B, latent_dim+2)
        raw_pars = self.decoder_net(de_input) ##(B, 27)

        ##apply physical constraint
        D, Ds, f, theta_par = self.physics_constraint(raw_pars)

        ##feeding into the IVIM bi-exponential function
        s_total, s_perf, s_diff = ivim_biex_signal(theta=theta_par, b_vals=self.bvalues)

        return s_total, (s_total, s_perf, s_diff), (theta_par, D, Ds, f)

    ##FORWARD PASS
    def forward(self, x, c):
        """
        x: (B, b-value, 3, 3)
        coord: (B, 2)
        returns: recon, mu, logvar, component, IVIM_parameters
        """
        mu, log_var = self.Encode(x)
        z = self.reparameterize(mu, log_var)
        recon, components, parameters = self.Decode(z, c)
        return recon, mu, log_var, components, parameters

    #RECONSTRUCTION LOSS WITH KL-DIVERGENCE
    def cvae_loss(self, recons, input_x, mu, logvar, params=None, kl_weight=1.0, param_reg_weight=0.01):
        # Critical checks against unnormalized or broken data feeds
        if torch.isnan(input_x).any() or torch.isnan(recons).any():
            print("NaN detected in input signals or reconstruction layer before loss calculation")

        ##Batch size
        batch_size = input_x.size(0)

        # Normalise in the dataset
        S0 = input_x[0:1, :, :].clone()
        x = input_x / ( S0 + 1e-8)   # divide by S(0) adding a non-zero terms for safe calculation

        #Reconstruction Loss
        recon_loss = F.mse_loss(recons, x, reduction='mean')

        #KL Divergence: Stabilized logvar constraints
        # Clamp logvar to prevent extreme values from causing inf/nan during logvar.exp()
        logvar = torch.clamp(logvar, min=-10.0, max=10.0)
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        kl_loss = kl_loss / batch_size

        ##Total loss
        loss = recon_loss + (kl_weight * kl_loss)

        #Physics Constraints
        if param_reg_weight > 0 and params is not None:
            theta, D, Ds, f = params

            f_prior = 0.20  
            f_reg = ((f - f_prior) ** 2).sum() / batch_size

            # Added safety margin to avoid compounding errors
            diff_reg = (F.relu(D - Ds * 0.30 + 1e-7) ** 2).sum() / batch_size

            smooth_d_v  = ((D[:, :, 1:, :] - D[:, :, :-1, :]) ** 2).sum()
            smooth_ds_v = ((Ds[:, :, 1:, :] - Ds[:, :, :-1, :]) ** 2).sum()
            smooth_d_h  = ((D[:, :, :, 1:] - D[:, :, :, :-1]) ** 2).sum()
            smooth_ds_h = ((Ds[:, :, :, 1:] - Ds[:, :, :, :-1]) ** 2).sum()
            
            smoothness = (smooth_d_v + smooth_ds_v + smooth_d_h + smooth_ds_h) / batch_size

            total_physics_penalty = (f_reg * 1.0) + (diff_reg * 10.0) + (smoothness * 0.1)
            loss += param_reg_weight * total_physics_penalty

        #If anything escapes as NaN, force a small valid scalar to keep training alive
        if torch.isnan(loss):
            loss = torch.tensor(1.0, requires_grad=True).to(x.device)

        return loss, recon_loss, kl_loss


In [ ]:
bvals_arr

## **TRAINING PIPELINE**

In [ ]:
##computing kl weights to optimize
def compute_kl_weight(epoch, warmup_epochs, anneal_epochs, kl_target):
    if epoch < warmup_epochs:
        return 0.0
    elif epoch < (warmup_epochs + anneal_epochs):
        progress = (epoch - warmup_epochs) / anneal_epochs
        return progress * kl_target
    else:
        return kl_target

##training function
def execute_cvae_training(model, train_loader, val_loader, num_epochs=100, lr=1e-4  , device="cuda"):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    ##constants 
    KL_TARGET = 0.01          
    PARAM_REG_WEIGHT = 0.1 ## 0.05    
    WARMUP_EPOCHS = 10 ##15       
    ANNEAL_EPOCHS = 30  ##35      

    print(f"Model Training Started...")
    print("_" * 100)

    for epoch in range(num_epochs):
        #TRAINING LOOP 
        model.train()
        ## KL annealing for KL weights
        current_kl_weight = compute_kl_weight(epoch, WARMUP_EPOCHS, ANNEAL_EPOCHS, KL_TARGET)

        ##training metrics
        train_loss_accum = 0.0 
        train_mse_accum = 0.0
        train_kl_accum = 0.0

        ##iterate over batches
        ## x_batch is image pixels, coord_batch is coordinates of spatial data
        for x_batch, coord_batch in train_loader:
            x_batch, coord_batch = x_batch.to(device), coord_batch.to(device)

            ##reset all gradients to zeros that is stored in the model's parameters
            optimizer.zero_grad()

            ##performing forward pass 
            recon_signals, mu, logvar, _, params = model(x_batch, coord_batch)

            ##computing loss
            loss, recon_loss, kl_loss = model.cvae_loss(
                recons=recon_signals, x=x_batch, mu=mu, logvar=logvar,
                params=params, kl_weight=current_kl_weight, param_reg_weight=PARAM_REG_WEIGHT
            )

            ##weight updates: backpropagation 
            loss.backward()
            ##computes total L2 norm and clips all the gradients to prevent them from the gradient exploding problem
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            ##Updates the model's parameters using the clipped gradients
            optimizer.step()
            
            train_loss_accum += loss.item()
            train_mse_accum += recon_loss.item()
            train_kl_accum += kl_loss.item()

        avg_train_loss = train_loss_accum / len(train_loader)
        avg_train_mse  = train_mse_accum / len(train_loader)
        avg_train_kl   = train_kl_accum / len(train_loader)

        #VALIDATION LOOP
        model.eval()
        val_loss_accum = 0.0
        val_mse_accum = 0.0
        val_kl_accum = 0.0
        val_mae_accum = 0.0

        ##no weight updates required
        with torch.no_grad():
            ##looping over validation batches
            for x_val, coord_val in val_loader:
                x_val, coord_val = x_val.to(device), coord_val.to(device)

                ##forward pass on validation data
                recon_v, mu_v, logvar_v, _, params_v = model(x_val, coord_val)

                ##computing loss
                v_loss, v_recon, v_kl = model.cvae_loss(
                    recons=recon_v, x=x_val, mu=mu_v, logvar=logvar_v, params=params_v,
                    kl_weight=current_kl_weight, param_reg_weight=PARAM_REG_WEIGHT
                )

                ##Mean Absolute Error (MAE) between reconstruction and input validation
                mae_metric = torch.mean(torch.abs(recon_v - x_val))
                
                val_loss_accum  += v_loss.item()
                val_mse_accum   += v_recon.item()
                val_kl_accum    += v_kl.item()
                val_mae_accum   += mae_metric.item()
                    
        avg_val_loss = val_loss_accum / len(val_loader)
        avg_val_mse  = val_mse_accum / len(val_loader)
        avg_val_kl   = val_kl_accum / len(val_loader)
        avg_val_mae  = val_mae_accum / len(val_loader)
        
        avg_val_rmse = np.sqrt(avg_val_mse)

        #prints metrices
        print(f"Epoch [{epoch+1:03d}/{num_epochs:03d}] | KL-Wt: {current_kl_weight:.1e}| "
              f"Train loss: {avg_train_loss:.4f}, MSE: {avg_train_mse:.4f}|| "
              f"Val loss: {avg_val_loss:.4f}, MSE: {avg_val_mse:.4f}, RMSE: {avg_val_rmse:.4f}, MAE: {avg_val_mae:.4f}")

    print("_" * 110)
    return model


## **Estimated IVIM Parameters**

In [ ]:
##IVIM parameters; D, D*, f
def generate_IVIM_params(model, mri_brain_data, device = "cuda"):
    model.eval()
    ##initialize
    all_D_maps = []
    all_Ds_maps = []
    all_f_maps = []

    ##no weights updates
    with torch.no_grad():
        for x_batch, x_coords in mri_brain_data:
            x_batch, x_coords = x_batch.to(device), x_coords.to(device)

            ##forward pass to extract spatial aware latent variable
            _, _, _, _,parameters = model(x_batch, x_coords)

            ##detach and move to cpu
            params = parameters.cpu().numpy()

            ##all three different IVIM parameters as array
            d_params  = params[..., 0]
            ds_params = params[..., 1]
            f_params  = params[..., 2]

            ##store in list 
            all_D_maps.append(d_params)
            all_Ds_maps.append(ds_params)
            all_f_maps.append(f_params)

    ##concatenating all parameters
    D   = np.concatenate(all_D_maps, axis=0)
    Ds  = np.concatenate(all_Ds_maps, axis=0)
    f   = np.concatenate(all_f_maps, axis=0)

    return D, Ds, f

In [ ]:
#b-value tensors from numpy array
bvalues_tensor = torch.from_numpy(bvals_arr.astype(np.float32)).float()
##Epoch
EPOCHS = 5
INPUT_CHANNEL = len(bvals_arr)
cvae_model = PhysicsCVAE(
    in_channel=INPUT_CHANNEL,  # 21 b-values
    bvalues=bvalues_tensor,
    latent_dim=64,
    coord_dim=2  # 2D normalized coordinates
)

# Move model to device first
cvae_model.to(device)

# Run the sanity-check training pass
execute_cvae_training(
    model=cvae_model,
    train_loader=dataloader_real,
    val_loader=dataloader_real,
    num_epochs=EPOCHS,
    lr=1e-3,
    device=device.type,
)

# **Test**

Finally, we instantiate the `PhysicsCVAE` model with the correct number of input channels (which is the number of b-values from your actual data) and train it using the `dataloader_real`.

### Updated Training Loop for GPU
Below is the modified training logic to ensure the `model` and `batches` are moved to the GPU memory.

In [ ]:
# ##IVIM parameters; D, D*, f
# def generate_IVIM_params(model, mri_brain_data, device = "cuda"):
#     model.eval()
#     ##initialize
#     all_D_maps = []
#     all_Ds_maps = []
#     all_f_maps = []

#     ##no weights updates
#     with torch.no_grad():
#         for x_batch, x_coords in mri_brain_data:
#             x_batch, x_coords = x_batch.to(device), x_coords.to(device)

#             ##forward pass to extract spatial aware latent variable
#             _, _, _, _,parameters = model(x_batch, x_coords)

#             ##detach and move to cpu
#             params = parameters.cpu().numpy()

#             ##all three different IVIM parameters as array
#             d_params  = params[..., 0]
#             ds_params = params[..., 1]
#             f_params  = params[..., 2]

#             ##store in list 
#             all_D_maps.append(d_params)
#             all_Ds_maps.append(ds_params)
#             all_f_maps.append(f_params)

#     ##concatenating all parameters
#     D   = np.concatenate(all_D_maps, axis=0)
#     Ds  = np.concatenate(all_Ds_maps, axis=0)
#     f   = np.concatenate(all_f_maps, axis=0)

#     return D, Ds, f

# #INFERENCE STEP: Generate the maps
# # Initialize empty volumes
# D_map = np.zeros(brain_msk.shape)
# Ds_map = np.zeros(brain_msk.shape)
# f_map = np.zeros(brain_msk.shape)

# model_real.eval()
# with torch.no_grad():
#     # We use a batch-based approach to fill the maps
#     # Using the same dataloader but mapping results back to original coordinates
#     # Note: We need a loader that also provides raw coordinates
#     inf_loader = DataLoader(dataset_real, batch_size=5000, shuffle=False)

#     # To accurately map back, we'll re-run a simple loop using the saved center_coords_real
#     # for memory efficiency and precision.
#     pointer = 0
#     for x_batch, c_batch in inf_loader:
#         x_batch_reshaped = x_batch.unsqueeze(2)
#         # Get mean parameters
#         mean_theta, _ = model_real.predict_parameters(x_batch_reshaped, c_batch, n_samples=10)

#         # Map batch back to 3D volume using the integer coordinates
#         for i in range(x_batch.size(0)):
#             z, y, x = center_coords_real[pointer]
#             D_map[z, y, x] = mean_theta[i, 0].item()
#             Ds_map[z, y, x] = mean_theta[i, 1].item()
#             f_map[z, y, x] = mean_theta[i, 2].item()
#             pointer += 1

# #VISUALIZATION
# #ADJUST RANGES HERE
# D_range = (0.0001, 0.004)     # min, max for Diffusion (D)
# Ds_range = (0.005, 0.020)      # min, max for Pseudo-diffusion (D*)
# f_range = (0.000, 0.300)        # min, max for Perfusion Fraction (f)

# slice_idx = 27

# # Mask the structural underlay itself so everything outside the brain is black
# raw_structural = b0_image[slice_idx, :, :]
# structural_mask = brain_msk[slice_idx] > 0
# structural_underlay = np.where(structural_mask, raw_structural, 0)

# def get_masked_map(pmap, mask):
#     # Mask out the background (zeros) for the color overlays
#     masked = np.ma.masked_where(mask[slice_idx] == 0, pmap[slice_idx])
#     return masked

# D_masked = get_masked_map(D_map, brain_msk)
# Ds_masked = get_masked_map(Ds_map, brain_msk)
# f_masked = get_masked_map(f_map, brain_msk)

# fig, axes = plt.subplots(1, 3, figsize=(20, 7), facecolor='black')

# def plot_overlay(ax, structural, parameter, title, cmap, vmin, vmax, label):
#     vmax_struct = np.percentile(structural[structural > 0], 98) if np.any(structural > 0) else 1.0
#     ax.imshow(structural, cmap='gray', vmax=vmax_struct)
#     im = ax.imshow(parameter, cmap=cmap, vmin=vmin, vmax=vmax, alpha=0.8)
#     ax.set_title(title, color='white', fontsize=14)
#     ax.axis('off')
#     cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
#     cbar.ax.yaxis.set_tick_params(color='white')
#     cbar.set_label(label, color='white')
#     plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color='white')

# # 1. D Map Overlay
# plot_overlay(axes[0], structural_underlay, D_masked,
#              f'D Map (Brain Only) - Slice {slice_idx}', 'inferno', D_range[0], D_range[1], 'mm²/s')

# # 2. D* Map Overlay
# plot_overlay(axes[1], structural_underlay, Ds_masked,
#              f'D* Map (Brain Only) - Slice {slice_idx}', 'inferno', Ds_range[0], Ds_range[1], 'mm²/s')

# # 3. f Map Overlay
# plot_overlay(axes[2], structural_underlay, f_masked,
#              f'f Map (Brain Only) - Slice {slice_idx}', 'inferno', f_range[0], f_range[1], 'Fraction')

# plt.tight_layout()
# plt.show()